# 00 - Generar Datos Sintéticos a Escala (Opcional)

Genera **datos adicionales** en el workspace para maximizar el procesamiento durante el taller.

> Ejecutar solo si desea más volumen de datos (~100K+ transacciones).


In [ ]:
%run ./00_variables


In [ ]:
import random
from datetime import datetime, timedelta

NUM_BATCHES = 10       # Ajustar según tiempo disponible
ROWS_PER_BATCH = 10000 # 10 batches x 10K = 100K filas extra

sucursales = ["SJ-001","SJ-002","AL-001","CA-001","HE-001","LI-001","PU-001","LI-002"]
productos = ["Cuenta Corriente","Cuenta Ahorro","Tarjeta Crédito","Préstamo Personal"]
canales = ["app_movil","sucursal","atm","web","sinpe"]
tipos = ["deposito","retiro","transferencia","pago","comision"]

output_path = f"{vol_path}/transacciones/extra"
dbutils.fs.mkdirs(output_path)

for batch in range(NUM_BATCHES):
    rows = []
    base_date = datetime(2026, 7, 1)
    for i in range(ROWS_PER_BATCH):
        ts = base_date + timedelta(minutes=random.randint(0, 43200))
        rows.append(f"TXN-EX{batch:02d}{i:06d},CTA-{random.randint(100000,999999)},CLI-{random.randint(10000,99999)},"
                    f"{ts.strftime('%Y-%m-%d %H:%M:%S')},{round(random.uniform(1000,5000000),2)},CRC,"
                    f"{random.choice(tipos)},{random.choice(canales)},{random.choice(sucursales)},{random.choice(productos)}")
    header = "transaccion_id,cuenta_id,cliente_id,fecha_hora,monto,moneda,tipo_transaccion,canal,sucursal_id,producto"
    content = header + "\n" + "\n".join(rows)
    dbutils.fs.put(f"{output_path}/extra_batch_{batch}.csv", content, overwrite=True)
    print(f"Batch {batch+1}/{NUM_BATCHES} generado ({ROWS_PER_BATCH} filas)")

print(f"\nTotal: {NUM_BATCHES * ROWS_PER_BATCH:,} transacciones extra en {output_path}")
